# Camada Silver — Tickets de Suporte

**Módulo 1 · Engenharia de Dados · Arquitetura Medalhão**

Este notebook transforma os dados brutos de `bronze.tb_suporte_tickets` em três tabelas Silver limpas e confiáveis:

| Tabela Silver | Origem Bronze | Descrição |
|---|---|---|
| `silver.tb_tickets_suporte` | `bronze.tb_suporte_tickets` | Tickets limpos, tipados e com colunas derivadas |
| `silver.dim_tipos_problema` | `bronze.tb_suporte_tickets` | Dimensão com tipos de problema normalizados |
| `silver.dim_agentes_suporte` | `bronze.tb_suporte_tickets` | Dimensão com métricas agregadas por agente |

---

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `tipo_problema` | Valores inconsistentes: abreviações, typos, variações de case (`pro`, `p3oduto`, `PRODUCT`, `DELAY`…) | Mapeamento para 4 categorias canônicas |
| `data_resolucao` | 2.072 nulos (tickets ainda abertos) | Mantidos como `null`; `resolvido = false` |
| `nota_avaliacao` | 2.072 nulos (sem avaliação para tickets abertos) | Mantidos como `null` |
| `tempo_resolucao_horas` | 2.072 nulos (tickets não resolvidos) | Mantidos como `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | Deduplicação por `ticket_id` |
| Datas | Formato ISO com timezone (`2023-01-03T05:13:00.000Z`) | Cast para `timestamp` |


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalogo       = 'vcommerce_catalog'
bronze_schema  = 'bronze'
silver_schema  = 'silver'

spark.sql(f'USE CATALOG {catalogo}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {silver_schema}')
spark.sql(f'USE SCHEMA {silver_schema}')

print(f'Catálogo : {catalogo}')
print(f'Schema   : {silver_schema}')

In [ ]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Usa a última ingestão de cada ticket (maior timestamp_ingestion) para
# garantir que reprocessamentos da Bronze não gerem duplicatas na Silver.

df_raw = spark.table(f'{bronze_schema}.tb_suporte_tickets')

window_dedup = Window.partitionBy('ticket_id').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   # será recriado com o instante desta carga
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

In [ ]:
# ─── Mapeamento canônico de tipo_problema ────────────────────────────────────
# Na Bronze foram encontradas 4 categorias reais fragmentadas em ~25 variações:
#
#   Entrega   → Entrega, ENTREGA, entrega, 3ntrega, entr, del, DELAY, delay
#   Reembolso → Reembolso, REEMBOLSO, reembolso, REFUND, ref, reemb, r3embolso
#   Produto   → Produto, PRODUTO, produto, pro, prod, p3oduto, PRODUCT, product
#   Pagamento → Pagamento, PAGAMENTO, pagamento, pag, pay, PAY, PAYMENT, payment,
#               p4gamento
#
# Estratégia: normaliza para lowercase e aplica mapeamento por prefixo/palavra-chave.
# Valores não mapeados são marcados como 'Outro' para investigação futura.

tipo_map = {
    # Entrega
    'entrega'  : 'Entrega',  '3ntrega' : 'Entrega',  'entr'  : 'Entrega',
    'del'      : 'Entrega',  'delay'   : 'Entrega',
    # Reembolso
    'reembolso': 'Reembolso','reemb'   : 'Reembolso','r3embolso': 'Reembolso',
    'refund'   : 'Reembolso','ref'     : 'Reembolso',
    # Produto
    'produto'  : 'Produto',  'pro'     : 'Produto',  'prod'  : 'Produto',
    'p3oduto'  : 'Produto',  'product' : 'Produto',
    # Pagamento
    'pagamento': 'Pagamento','pag'     : 'Pagamento','p4gamento': 'Pagamento',
    'pay'      : 'Pagamento','payment' : 'Pagamento',
}

# Constrói expressão CASE WHEN a partir do dicionário
tipo_expr = F.col('tipo_problema')
for raw_val, canonical in tipo_map.items():
    tipo_expr = F.when(
        F.lower(F.col('tipo_problema')) == raw_val, canonical
    ).otherwise(tipo_expr)

# Valores que não bateram com nenhum mapeamento ficam como 'Outro'
known_lower = [k.lower() for k in tipo_map.keys()]
tipo_expr = F.when(
    F.lower(F.col('tipo_problema')).isin(known_lower), tipo_expr
).otherwise(F.lit('Outro'))

print('Expressão de mapeamento criada.')

In [ ]:
# ─── Transformações principais ───────────────────────────────────────────────

df_silver = (
    df_dedup

    # 1. Tipos corretos para datas (ISO 8601 com timezone)
    .withColumn('data_abertura',   F.to_timestamp('data_abertura'))
    .withColumn('data_resolucao',  F.to_timestamp('data_resolucao'))

    # 2. Normalização de tipo_problema
    .withColumn('tipo_problema', tipo_expr)

    # 3. Colunas derivadas de negócio
    .withColumn(
        'resolvido',
        F.col('data_resolucao').isNotNull()
    )
    .withColumn(
        'hora_abertura',
        F.hour('data_abertura').cast('int')
    )
    .withColumn(
        'dia_semana_abertura',
        F.date_format('data_abertura', 'EEEE')   # nome completo em inglês; adaptar locale se necessário
    )

    # 4. Tipos numéricos
    .withColumn('tempo_resolucao_horas', F.col('tempo_resolucao_horas').cast('decimal(10,2)'))
    .withColumn('nota_avaliacao',        F.col('nota_avaliacao').cast('decimal(3,1)'))

    # 5. Marca temporal desta carga Silver
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 6. Ordenação de colunas conforme schema Silver acordado
    .select(
        'ticket_id',
        'id_cliente',
        'id_pedido',
        'tipo_problema',
        'data_abertura',
        'data_resolucao',
        'tempo_resolucao_horas',
        'agente_suporte',
        'nota_avaliacao',
        'resolvido',
        'hora_abertura',
        'dia_semana_abertura',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

In [ ]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────

total = df_silver.count()
resolvidos     = df_silver.filter(F.col('resolvido') == True).count()
nao_resolvidos = df_silver.filter(F.col('resolvido') == False).count()
outros_tipo    = df_silver.filter(F.col('tipo_problema') == 'Outro').count()

print(f'Total de tickets       : {total:,}')
print(f'Resolvidos             : {resolvidos:,} ({resolvidos/total*100:.1f}%)')
print(f'Abertos (sem resolução): {nao_resolvidos:,} ({nao_resolvidos/total*100:.1f}%)')
print(f'Tipo "Outro" (suspeitos): {outros_tipo:,}')
print()
print('Distribuição de tipo_problema após normalização:')
df_silver.groupBy('tipo_problema').count().orderBy(F.desc('count')).show()

In [ ]:
# ─── Grava silver.tb_tickets_suporte ─────────────────────────────────────────
# overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable('silver.tb_tickets_suporte')
)

print(f'✓ silver.tb_tickets_suporte gravada com {df_silver.count():,} registros.')

In [ ]:
# ─── dim_tipos_problema ───────────────────────────────────────────────────────
# Dimensão com os 4 tipos canônicos e sua categoria de negócio.
#
# Categoria derivada:
#   Entrega   → Logística
#   Reembolso → Financeiro
#   Produto   → Qualidade
#   Pagamento → Financeiro
#   Outro     → Indefinido

df_dim_tipos = (
    spark.table('silver.tb_tickets_suporte')
         .select('tipo_problema')
         .distinct()
         .withColumn(
             'categoria_problema',
             F.when(F.col('tipo_problema') == 'Entrega',   'Logística')
              .when(F.col('tipo_problema') == 'Reembolso', 'Financeiro')
              .when(F.col('tipo_problema') == 'Produto',   'Qualidade')
              .when(F.col('tipo_problema') == 'Pagamento', 'Financeiro')
              .otherwise('Indefinido')
         )
         .orderBy('tipo_problema')
)

df_dim_tipos.show()

(
    df_dim_tipos.write
                .format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable('silver.dim_tipos_problema')
)

print('✓ silver.dim_tipos_problema gravada.')

In [ ]:
# ─── dim_agentes_suporte ──────────────────────────────────────────────────────
# Dimensão com métricas agregadas por agente, derivadas de tb_tickets_suporte.
#
# Métricas calculadas:
#   qtd_tickets_resolvidos  → total de tickets onde resolvido = true
#   nota_media_atendimento  → média de nota_avaliacao (ignora nulos)

df_dim_agentes = (
    spark.table('silver.tb_tickets_suporte')
         .groupBy('agente_suporte')
         .agg(
             F.count(
                 F.when(F.col('resolvido') == True, 1)
             ).alias('qtd_tickets_resolvidos'),
             F.round(
                 F.avg('nota_avaliacao'), 2
             ).alias('nota_media_atendimento'),
         )
         .withColumn(
             'nota_media_atendimento',
             F.col('nota_media_atendimento').cast('decimal(4,2)')
         )
         .orderBy(F.desc('qtd_tickets_resolvidos'))
)

df_dim_agentes.show(truncate=False)

(
    df_dim_agentes.write
                  .format('delta')
                  .mode('overwrite')
                  .option('overwriteSchema', 'true')
                  .saveAsTable('silver.dim_agentes_suporte')
)

print('✓ silver.dim_agentes_suporte gravada.')

In [ ]:
# ─── Resumo final ─────────────────────────────────────────────────────────────

tabelas = [
    'silver.tb_tickets_suporte',
    'silver.dim_tipos_problema',
    'silver.dim_agentes_suporte',
]

print('=== Camada Silver — Tickets de Suporte ===')
print(f'{"Tabela":<35} {"Linhas":>8} {"Colunas":>8}')
print('-' * 55)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<35} {df_t.count():>8,} {len(df_t.columns):>8}')